# Dataset analysis for Caltech101 and Enrico

This notebook keeps the code compact and focuses on a few paper-friendly views: Caltech101 summary statistics, then Enrico screenshots vs wireframes with UMAP-based similarity and overlap metrics.

In [ ]:
!pip install -q umap-learn

import numpy as np
import seaborn as sns
from IPython.display import display
from torch.utils.data import ConcatDataset, DataLoader
from torchvision import datasets

from natural_images import (
    AugmentationWrapper, 
    make_caltech_base_transform, 
    stratified_three_way_split
)
from screen_images import (
    CustomEnricoDataset, 
    make_screen_base_transform
)
from dataset_analysis_utils import (
    extract_labels,
    class_count_frame,
    split_summary_frame,
    plot_class_count_histogram,
    show_sample_grid,
    dataset_to_matrix,
    project_with_umap,
    plot_umap,
    summarize_umap,
    print_umap_summary,
)
from training import (
    mean_and_std_for_normalization, 
    seed_everything
)


In [ ]:
SEED = 317
seed_everything(seed=SEED)
sns.set_theme(style='whitegrid')

In [ ]:
screen_base_transform = make_screen_base_transform(resize=(300, 200))
wireframe_base_transform = screen_base_transform

## Caltech101 summary

The goal here is to keep the stats compact but useful: split balance, class balance, a few sample images, and the original-train mean/std used for standardization.

In [ ]:
caltech_dataset = datasets.Caltech101(root='./data', download=True)
caltech_train_raw, caltech_val_raw, caltech_test_raw = stratified_three_way_split(
    dataset=caltech_dataset,
    train_ratio=0.7,
    val_ratio=0.15,
    test_ratio=0.15,
    random_state=SEED,
)

caltech_labels = extract_labels(caltech_dataset)
caltech_class_names = [str(index) for index in sorted(np.unique(caltech_labels))]
caltech_train_counts = class_count_frame(caltech_train_raw, caltech_class_names)
caltech_split_counts = split_summary_frame(caltech_train_raw, caltech_val_raw, caltech_test_raw)

print(f'Caltech101 total samples: {len(caltech_dataset)}')
print(f'Caltech101 classes: {len(caltech_class_names)}')
display(caltech_split_counts)
display(caltech_train_counts['count'].describe().to_frame('train_class_counts'))
print('Most frequent training classes')
display(caltech_train_counts.sort_values('count', ascending=False).head(10))
print('Least frequent training classes')
display(caltech_train_counts.sort_values('count', ascending=True).head(10))
plot_class_count_histogram(caltech_train_counts['count'], 'Caltech101 training class frequency distribution')
show_sample_grid(caltech_train_raw, class_names=caltech_class_names, title='Caltech101 train samples', n_samples=9)

caltech_base_transform = make_caltech_base_transform(resize=(300, 200))
caltech_train_loader = DataLoader(
    AugmentationWrapper(caltech_train_raw, caltech_base_transform),
    batch_size=128,
    shuffle=False,
    num_workers=0,
)
caltech_mean, caltech_std = mean_and_std_for_normalization(caltech_train_loader)
print('Caltech101 train mean:', np.round(caltech_mean, 6))
print('Caltech101 train std:', np.round(caltech_std, 6))

## Enrico screenshots and wireframes

For UMAP, the notebook uses the raw split first, then the same pixel-level embedding pipeline for screenshots and wireframes so the two modalities stay comparable.

In [ ]:
ENRICO_ROOT = '/kaggle/input/datasets/nazariyyuchnovskiy/enricoscreenshotsandwireframes'

screen_train_raw, screen_val_raw, screen_test_raw = CustomEnricoDataset.create_splits(
    root=ENRICO_ROOT,
    seed=SEED,
    use_wireframes=False,
    transform=None,
    train_transform=None,
    eval_transform=None,
    augment_for_each=None,
    allowed_classes=None,
)
wire_train_raw, wire_val_raw, wire_test_raw = CustomEnricoDataset.create_splits(
    root=ENRICO_ROOT,
    seed=SEED,
    use_wireframes=True,
    transform=None,
    train_transform=None,
    eval_transform=None,
    augment_for_each=None,
    allowed_classes=None,
)

screen_all = ConcatDataset([screen_train_raw, screen_val_raw, screen_test_raw])
wire_all = ConcatDataset([wire_train_raw, wire_val_raw, wire_test_raw])
screen_class_names = [name for name, index in sorted(screen_train_raw.class_to_idx.items(), key=lambda item: item[1])]
wire_class_names = [name for name, index in sorted(wire_train_raw.class_to_idx.items(), key=lambda item: item[1])]

print(f'Screenshot split sizes: {len(screen_train_raw)} / {len(screen_val_raw)} / {len(screen_test_raw)}')
print(f'Wireframe split sizes: {len(wire_train_raw)} / {len(wire_val_raw)} / {len(wire_test_raw)}')
display(split_summary_frame(screen_train_raw, screen_val_raw, screen_test_raw))
display(split_summary_frame(wire_train_raw, wire_val_raw, wire_test_raw))
display(class_count_frame(screen_all, screen_class_names).sort_values('count', ascending=False).head(10))
display(class_count_frame(wire_all, wire_class_names).sort_values('count', ascending=False).head(10))

In [ ]:
screen_features, screen_labels = dataset_to_matrix(screen_all, resize=(48, 48))
wire_features, wire_labels = dataset_to_matrix(wire_all, resize=(48, 48))

screen_coords = project_with_umap(screen_features)
wire_coords = project_with_umap(wire_features)

plot_umap(screen_coords, screen_labels, screen_class_names, 'Enrico screenshots UMAP')
plot_umap(wire_coords, wire_labels, wire_class_names, 'Enrico wireframes UMAP')

screen_summary = summarize_umap(screen_coords, screen_labels, screen_class_names, k=15)
wire_summary = summarize_umap(wire_coords, wire_labels, wire_class_names, k=15)

print_umap_summary(screen_summary, 'Screenshots UMAP summary')
print_umap_summary(wire_summary, 'Wireframes UMAP summary')

## Useful follow-ups for the paper

A few extra visuals or metrics that would fit this notebook without making it heavy: a centroid-distance heatmap, a nearest-neighbor retrieval panel for the most overlapping classes, silhouette or Davies-Bouldin scores per modality, and a compact class-count chart for Caltech101.